NOTEBOOK 5: DBSCAN Outlier Detection
Run AFTER Notebook 4 is complete.
 
Inputs:
    - analytical_dataset.csv
    - hypothesis_results.csv
 
Outputs:
    - ticker_sectors.csv              (sector lookup — pulled once from yfinance)
    - politician_profiles.csv         (one row per politician, all DBSCAN features)
    - dbscan_results.csv              (clustering labels + outlier tier)
    - convergence_politicians.csv     (politicians flagged by BOTH Wilcoxon + DBSCAN)
 
Methodology:
    - Spark for data loading and aggregation
    - yfinance for sector enrichment (run once, cached to disk)
    - sklearn DBSCAN with epsilon from k-distance elbow plot
    - PCA 2D projection for visualisation

# CELL 1 — Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os
import time
from pathlib import Path

warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import yfinance as yf

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

# CONFIG (OS-independent paths)
OUTPUT_DIR = Path("data") / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYTICAL_PATH = OUTPUT_DIR / "analytical_dataset.csv"
HYPOTHESIS_PATH = OUTPUT_DIR / "hypothesis_results.csv"
SECTORS_PATH    = OUTPUT_DIR / "ticker_sectors.csv"
PROFILES_PATH   = OUTPUT_DIR / "politician_profiles.csv"
DBSCAN_OUT      = OUTPUT_DIR / "dbscan_results.csv"
CONVERGENCE_OUT = OUTPUT_DIR / "convergence_politicians.csv"

MIN_TRADES_DBSCAN  = 10   # lower than Wilcoxon — DBSCAN works on profiles
                        # not individual trades so 10 is okay
DBSCAN_MIN_SAMPLES = 5    # minimum cluster size

# Committee → sector mapping
# Based on publicly available committee jurisdiction information
COMMITTEE_SECTOR_MAP = {
    'Armed Services':          ['Industrials', 'Aerospace & Defense'],
    'Financial Services':      ['Financials', 'Financial Services'],
    'Energy and Commerce':     ['Energy', 'Utilities', 'Healthcare'],
    'Science and Technology':  ['Technology', 'Communication Services'],
    'Agriculture':             ['Consumer Staples', 'Materials'],
    'Transportation':          ['Industrials', 'Energy'],
    'Judiciary':               [],
    'Foreign Affairs':         ['Energy', 'Industrials'],
    'Intelligence':            ['Technology', 'Industrials'],
    'Finance':                 ['Financials', 'Financial Services'],
    'Banking':                 ['Financials', 'Financial Services'],
    'Commerce':                ['Technology', 'Communication Services',
                                'Consumer Discretionary'],
    'Health':                  ['Healthcare'],
    'Environment':             ['Energy', 'Utilities', 'Materials'],
}

print("All imports loaded successfully")

All imports loaded successfully


# CELL 2 — Start Spark & Load Data

Run this cell below if Cell 2 fails. Or to play safe, just run this cell together.

In [ ]:
import os

# 1. Update package lists so it can find the packages
!sudo apt-get update -qq

# 2. Install Java 17 instead of 11
!sudo apt-get install openjdk-17-jdk-headless -qq > /dev/null

# 3. Set the JAVA_HOME environment variable to the Java 17 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [ ]:
spark = SparkSession.builder \
    .appName("SC2320_Project_Group6_DBSCAN") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
 
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
 
# Load analytical dataset
sdf = spark.read.csv(str(ANALYTICAL_PATH), header=True, inferSchema=True)
print(f"Analytical dataset: {sdf.count():,} rows")
 
# Load hypothesis results
hyp_df = pd.read_csv(HYPOTHESIS_PATH)
print(f"Hypothesis results: {len(hyp_df):,} politicians")
 
# Significant politicians from Notebook 4
sig_politicians = set(
    hyp_df[hyp_df['significant'] == True]['politician'].tolist()
)
print(f"Significant politicians from Notebook 4: {len(sig_politicians)}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/21 09:13:47 WARN Utils: Your hostname, codespaces-a752f8, resolves to a loopback address: 127.0.0.1; using 10.0.0.58 instead (on interface eth0)
26/04/21 09:13:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 09:13:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


Analytical dataset: 24,130 rows
Hypothesis results: 75 politicians
Significant politicians from Notebook 4: 12


# CELL 3 — Pull Sector Data from yfinance (run once, cached)

In [ ]:
print("=" * 60)
print("SECTOR ENRICHMENT via yfinance")
print("=" * 60)
 
if os.path.exists(SECTORS_PATH):
    print(f"Cached sector file found — loading {SECTORS_PATH}")
    sectors_df = pd.read_csv(SECTORS_PATH)
else:
    print("No cache found — pulling sectors from yfinance...")
    print("This runs once only and saves to disk.\n")
 
    # Get unique tickers from analytical dataset
    tickers_pdf = sdf.select('ticker').distinct().toPandas()
    unique_tickers = tickers_pdf['ticker'].dropna().unique().tolist()
    print(f"Unique tickers to enrich: {len(unique_tickers):,}")
 
    sector_records = []
    failed_tickers = []
 
    for i, ticker in enumerate(unique_tickers):
        if i % 100 == 0:
            print(f"  [{i:>4}/{len(unique_tickers)}] processing...")
        try:
            info   = yf.Ticker(ticker).info
            sector = info.get('sector', 'Unknown')
            industry = info.get('industry', 'Unknown')
            sector_records.append({
                'ticker':   ticker,
                'sector':   sector if sector else 'Unknown',
                'industry': industry if industry else 'Unknown',
            })
        except Exception:
            failed_tickers.append(ticker)
            sector_records.append({
                'ticker':   ticker,
                'sector':   'Unknown',
                'industry': 'Unknown',
            })
        time.sleep(0.3)   # polite delay
 
    sectors_df = pd.DataFrame(sector_records)
    sectors_df.to_csv(SECTORS_PATH, index=False)
    print(f"\nSectors saved to {SECTORS_PATH}")
    print(f"Failed tickers: {len(failed_tickers)}")
 
# Replace Unknown/None with 'Other'
sectors_df['sector'] = sectors_df['sector'].fillna('Other')
sectors_df.loc[sectors_df['sector'] == 'Unknown', 'sector'] = 'Other'
 
print(f"\nSector distribution:")
print(sectors_df['sector'].value_counts().to_string())

SECTOR ENRICHMENT via yfinance


NameError: name 'os' is not defined

# CELL 4 — Spark: Merge Sector Into Analytical Dataset

In [ ]:
# Upload sectors to Spark
sectors_spark = spark.createDataFrame(sectors_df[['ticker', 'sector']])
 
# Join sector onto analytical dataset
sdf_enriched = sdf.join(sectors_spark, on='ticker', how='left')
sdf_enriched = sdf_enriched.fillna({'sector': 'Other'})
 
print(f"Enriched dataset rows: {sdf_enriched.count():,}")
sdf_enriched.select('politician', 'ticker', 'sector',
                    'abn_ret_mean', 'anomaly_flag_L1').show(5, truncate=False)